# XRD phase identification

Identifies every crystalline phase in a powder XRD scan, minor phases near 1 wt% included, and fits their lattice parameters — automatically, from a list of elements. Reference structures come from a local library first, then COD and Materials Project, plus optional MOF collections (CoRE MOF 2024, QMOF) and your own CIFs (CSD, Published, local).

**Preset:** IUCr CPD round-robin sample 1a (`examples/cpd-1a.txt`: fluorite + zincite + corundum), which runs offline with the bundled `reference_library`. Settings for battery cathodes (NMC) and broad-peak MOFs are in the comments of the settings cell.

All code is in `xrd_refs.py`; **restart the kernel after replacing it** and check `X.__version__`.


In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
sys.path.insert(0, str(Path.cwd()))
import xrd_refs as X
print("xrd_refs version:", X.__version__)

# ---------------- settings ----------------
LIBRARY = Path(r"reference_library")          # ONE folder for all projects

EXPERIMENTAL_FILE = r"examples/cpd-1a.txt"    # <- your scan (2theta, intensity)
OUT        = Path("results/cpd-1a")           # tables, plots and reference CIFs for this sample

ELEMENTS   = ["Al", "Ca", "F", "Zn", "O"]     # all elements in the sample (H included!)
SYSTEMS    = [["Al", "O"], ["Ca", "F"], ["Zn", "O"]]   # phases' chemical systems you expect; None if unknown
WAVELENGTH = "CuKa"      # "CuKa": lab Cu tube, Ka1+Ka2 doublet   "CuKa1": Ka1 only / synchrotron converted
SIM_RANGE  = (5, 150)

# which sources to use (True = on). Off = neither their library files nor online searches.
DATABASES = {
    "COD":       True,    # Crystallography Open Database (online + library/COD)
    "MP":        True,    # Materials Project (needs MP_API_KEY for online searches; almost no MOFs)
    "CoRE_MOF":  False,   # CoRE MOF 2024, experimental MOF structures (import once, section 0)
    "QMOF":      False,   # QMOF, DFT-optimised MOFs (large download; cells can be 1-3 % off)
    "CSD":       False,   # CIFs you downloaded from the CCDC -> library/CSD
    "Published": False,   # CIFs from papers' SI / Zenodo / Mendeley -> library/Published
    "local":     True,    # your own CIFs / models -> library/local
}

GOOD_SCORE, GOOD_EXPLAINED_PCT, MAX_CELL_SHIFT_PCT = 0.85, 80.0, 1.0
MIXTURE    = True        # judge a stage by all identified phases together
USE_GENERATED_MODELS = False                  # True: add built NMC/LiCoO2 models (battery samples)
IMPURITY_LIBRARY     = []                     # COD side-phase searches (needs COD on):
                                              # a name from X.IMPURITY_LIBRARIES, a list of systems, or []
# NMC example: ELEMENTS = ["Li","Ni","Mn","Co","O"], SYSTEMS = None, WAVELENGTH = "CuKa1", SIM_RANGE = (10, 80),
#              USE_GENERATED_MODELS = True, IMPURITY_LIBRARY = "Li-ion layered oxide (NMC/NCA/LCO)"
# MOF example (Cu-HHTP): ELEMENTS = ["Cu","C","H","O"], SYSTEMS = [["Cu","C","H","O"]], SIM_RANGE = (2, 60),
#              GOOD_* = 0.60, 70.0, 1.5, DATABASES: COD, CoRE_MOF, QMOF, Published on, MP off,
#              IMPURITY_LIBRARY = [["Cu","O"], ["Cu"], ["Cu","O","H"]]

MP_API_KEY = "".strip()
if not MP_API_KEY:
    MP_API_KEY = os.environ.get("MP_API_KEY", "")
print("Materials Project key:", f"set ({len(MP_API_KEY)} characters)" if MP_API_KEY else "NOT set (library + COD only)")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Optional speed settings
# X.WORKERS = 4                    # CPU cores to use (default: all but one, at most 12); 1 = single core
# X.set_cache_dir(r"path/to/cache") # where the disk cache goes (default: <LIBRARY>/_cache)
# X.clear_cache()                  # delete the cache for a clean rebuild
# X.PEAK_WIDTH = 0.5               # force the matching peak width (deg); default is automatic


## 0. (Once) put the MOF collections into the library
Set an entry of `IMPORT_NOW` to `True`, run the cell once, then set it back to `False`. CoRE MOF is about 44 MB (8 300 structures); QMOF is a much larger download. The archives are kept in `library/_downloads`, so a re-import does not download again.

For CIFs you get yourself — a CCDC deposition (Access Structures, e.g. CCDC 2518426), a paper's supporting information, a Zenodo or Mendeley zip — save them in a folder or keep the zip, and import them into `CSD` or `Published` with `source=`. You can also simply copy CIFs into `library/CSD` or `library/Published`: new files are indexed automatically at the next run.

In [ ]:
IMPORT_NOW = {"CoRE_MOF": False, "QMOF": False}      # True once to download + index, then back to False
for name, do in IMPORT_NOW.items():
    if do:
        X.import_collection(name, LIBRARY)

# your own downloads (folder of CIFs or a zip) - uncomment and edit the path:
# X.import_collection("CSD", LIBRARY, source=r"path\to\CCDC_download_folder")
# X.import_collection("Published", LIBRARY, source=r"path\to\SI_cifs.zip")

display(X.collection_status(LIBRARY))

## 1. Find references: library first, databases only if needed
With `SYSTEMS` given, any expected system missing from the library is fetched first (one COD query each), then the library is judged. The log shows each stage and why it stopped; for a mixture it also lists the phases and how much of the pattern they explain together.

In [ ]:
built_refs = []
if USE_GENERATED_MODELS:
    built_refs = [X.build_layered_preset(n) for n in ["LiCoO2", "NMC111", "NMC532", "NMC622", "NMC811"]]

x, y = X.load_xy(EXPERIMENTAL_FILE)
print(f"Scan: {len(x)} points, {x.min():.2f}-{x.max():.2f} deg, step {np.median(np.diff(x)):.3f}\n")

all_refs, ranking, search_log = X.auto_reference_search(
    ELEMENTS, x, y, LIBRARY, WAVELENGTH, mp_api_key=MP_API_KEY, extra_refs=built_refs,
    systems=SYSTEMS, mixture=MIXTURE, databases=DATABASES,
    min_score=GOOD_SCORE, min_explained_pct=GOOD_EXPLAINED_PCT, max_cell_shift_pct=MAX_CELL_SHIFT_PCT)

search_log.to_csv(OUT / "search_log.csv", index=False)
ranking.to_csv(OUT / "ranking_vs_experiment.csv", index=False)
pd.set_option("display.max_colwidth", 120)
display(search_log)
display(ranking[["source", "kind", "ref_id", "formula", "spacegroup", "score", "cell_scale_pct",
                 "explained_pct", "unexplained_peaks", "missing_strong"]].head(12))

## 2. Summary of the best-matching references

In [ ]:
by_id = {r["ref_id"]: r for r in all_refs}
table = X.summary_table([by_id[i] for i in ranking.ref_id.head(60)], SIM_RANGE, WAVELENGTH)
table.to_csv(OUT / "reference_summary.csv", index=False)
display(table[[c for c in ["source", "kind", "ref_id", "formula", "spacegroup", "a", "c", "ordered", "group",
                           "strongest_peaks"] if c in table.columns]].head(15))

## 3. Which phases are in the sample?
Cullity's iterative procedure: main phase, then phases explaining the leftover peaks. A minor phase is accepted when it explains at least 3 leftover peaks (its pattern score stays low because only its strongest lines rise above the noise). Isolated single peaks are listed, never assigned.

In [ ]:
if IMPURITY_LIBRARY and DATABASES.get("COD", True):
    impurity_refs = X.search_impurities(IMPURITY_LIBRARY, out_dir=LIBRARY)
else:
    impurity_refs = []
phases, report = X.identify_phases(all_refs, x, y, WAVELENGTH, impurity_candidates=impurity_refs)
report.to_csv(OUT / "phase_identification.csv", index=False)
pd.set_option("display.max_colwidth", 200)
display(report)

# copy the CIFs of the identified phases and the 10 best candidates to OUT/reference_cifs (with sources.csv)
X.save_reference_cifs(ranking, all_refs + impurity_refs, OUT, n_top=10, phases=phases)


## 4. Lattice parameters of every identified phase
The main phase is fitted with a free 2θ zero offset; the other phases reuse that offset (it is the same for the whole scan, and a minor phase has too few peaks to fit both). Only peaks that no other identified phase could explain are used. Monoclinic/triclinic phases are skipped.

In [ ]:
cells, zero = [], None
for i, p in enumerate(phases):
    try:
        res, used = X.fit_cell(x, y, p, WAVELENGTH, other_phases=phases, zero_shift=zero)
    except Exception as ex:
        print(f"{p['formula']} ({p['ref_id']}): not fitted - {ex}")
        continue
    if i == 0:
        zero = res["zero_shift_deg"]
    cells.append(res)
    print(f"{p['formula']} {res['system']}: a = {res['a']:.4f} A" + (f", c = {res['c']:.4f} A" if res['system'] != 'cubic' else "")
          + f"  ({res['n_peaks']} peaks, rms {res['rms_deg']:.4f} deg, zero {res['zero_shift_deg']:+.4f} deg"
          + (" fixed" if res["zero_fixed"] else "") + ")" + (f"  WARNING: {res['warning']}" if "warning" in res else ""))
cells = pd.DataFrame(cells)
cells.to_csv(OUT / "lattice_parameters.csv", index=False)
display(cells.round(5))

## 5. Plot: your scan with the identified phases

In [ ]:
by_id.update({p["ref_id"]: p for p in phases})
pick = [p["ref_id"] for p in phases]
for rid in X.pick_for_plot(ranking, n_top=2, per_source=1, table=table):
    if rid not in pick and len(pick) < 6:
        pick.append(rid)
roles = dict(zip(report.ref_id, report.role))
score = dict(zip(ranking.ref_id, ranking.score))
for rid in pick:
    r = by_id[rid]
    r["plot_label"] = f"{r['formula']} [{r['spacegroup']}] {rid} · {roles.get(rid, 'score %.2f' % score.get(rid, float('nan')))}"
peaks = X.find_experimental_peaks(x, X.subtract_background(x, y)[0])
fwhm = float(np.clip(peaks.nlargest(5, "rel_intensity").fwhm.median(), 0.02, 0.3))
plot_range = (float(np.floor(x.min())), float(np.ceil(x.max())))
left = X.unexplained_peaks(phases, x, y, WAVELENGTH)    # peaks no identified phase explains -> red guide lines
ax = X.plot_stack([by_id[i] for i in pick], plot_range, WAVELENGTH, fwhm=fwhm,
                  experimental=(x, y), unexplained=left.two_theta)
ax.set_title("Identified phases")
plt.tight_layout(); plt.savefig(OUT / "phases.png", dpi=200); plt.show()

# zoom: intensity axis cut low so ~1 % peaks show; each phase's reflections as a row of ticks below the data
fig, ax = plt.subplots(figsize=(11, 4.2))
base = np.percentile(y, 5)
top = np.median(y) + 0.02 * (y.max() - y.min())
band = 0.09 * (top - base)
ax.plot(x, y, color="black", lw=0.7)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for j, p in enumerate(phases):
    pk_ = X.simulate_peaks(p["structure"], plot_range, X._emission_lines(WAVELENGTH)[0][0])
    pk_ = pk_[pk_.intensity >= 2]
    y0 = base - (j + 1) * band
    ax.vlines(pk_.two_theta, y0, y0 + 0.75 * band, color=colors[j % len(colors)], lw=1)
    ax.text(plot_range[0] + 0.5, y0 + 0.35 * band, f" {p['formula']}", color=colors[j % len(colors)], fontsize=8, va="center")
ax.set_ylim(base - (len(phases) + 0.3) * band, top)
ax.set_xlim(*plot_range); ax.set_xlabel(r"2$\theta$ / degrees"); ax.set_ylabel("counts (zoomed)")
ax.set_title("Zoom on weak peaks; ticks = reflections of each identified phase (Ka1, >= 2 %)")
plt.tight_layout(); plt.savefig(OUT / "phases_zoom.png", dpi=200); plt.show()